# Pairwise Separability Index (PSI)

Measures how clearly the clusters differ from each other on each interpretation variable:

- ordered variables (Age, MMSE, nWBV, eTIV, Education, SES, CDR): |Cliff's δ|, with
  δ = P(X_i > X_j) − P(X_i < X_j) for one patient from cluster i and one from cluster j;
- sex: Total Variation Distance between the two clusters, TVD = ½ Σ |p_ic − p_jc|.

Both lie between 0 and 1. They are averaged over all pairs of clusters (per-variable PSI) and then over the
variables (aggregate PSI). Thresholds: negligible < 0.147 ≤ small < 0.33 ≤ medium < 0.474 ≤ large.
Unlike the silhouette, the PSI is computed on the same variables whatever distance a configuration used, so
configurations with different distances can be compared.

The code below was written by Fabio Scielzo-Ortiz and is used without changes, split into cells.

Imports

In [ ]:
import numpy as np
import pandas as pd
from itertools import combinations
from dataclasses import dataclass, field
from typing import Sequence, Literal, Union

### Cliff's δ

Computed from ranks, so tied values (common in MMSE, CDR, Education and SES) are handled correctly.

In [ ]:
def cliffs_delta(x: np.ndarray, y: np.ndarray) -> float:
    """
    Cliff's delta between two independent samples x and y.
    delta = P(X > Y) - P(X < Y), in [-1, 1].
    Non-parametric, robust to ordinal/Likert-type data, does not assume
    equal variances or normality. Computed via the rank-based Mann-Whitney
    shortcut (avoids the O(n*m) double loop for large n).
    """
    x = np.asarray(x)
    y = np.asarray(y)
    nx, ny = len(x), len(y)

    all_vals = np.concatenate([x, y])
    ranks = pd.Series(all_vals).rank(method="average").to_numpy()
    rank_x = ranks[:nx]

    r_sum_x = rank_x.sum()
    u_x = r_sum_x - nx * (nx + 1) / 2
    delta = (2 * u_x) / (nx * ny) - 1
    return float(np.clip(delta, -1.0, 1.0))

### Total Variation Distance

In [ ]:
def total_variation_distance(p: np.ndarray, q: np.ndarray) -> float:
    """
    Total Variation Distance between two categorical (probability) distributions.
    TVD = 0.5 * sum(|p_i - q_i|), in [0, 1].
    0 = identical category-proportion profiles; 1 = disjoint supports.
    This is the natural nominal-variable analogue of Cliff's delta: instead
    of comparing the ordering of individual observations, it compares how
    differently the two clusters distribute their mass across categories
    (e.g. discourse_tone: Hostile/Analytical/Emotional/...). Equivalent (up
    to the 0.5 factor) to the L1 distance between empirical category
    distributions, and closely related to the overlap coefficient
    (1 - TVD = proportion of shared/overlapping category mass).
    """
    cats = sorted(set(p.index) | set(q.index))
    p_full = p.reindex(cats, fill_value=0.0)
    q_full = q.reindex(cats, fill_value=0.0)
    return float(0.5 * np.abs(p_full - q_full).sum())

### TVD for every pair of clusters (nominal variable)

In [ ]:
def pairwise_separability_nominal(
    data: pd.DataFrame,
    cluster_col: str,
    variable: str,
) -> "PairwiseVariableResult":
    """
    For a nominal variable, computes Total Variation Distance for every
    pair of clusters (how differently they distribute mass across
    categories), and the simple proportion mismatch in the *dominant*
    category (1 if the two clusters have a different modal category, 0
    otherwise) as an easily-readable companion statistic.
    """
    groups = sorted(data[cluster_col].unique())
    pair_results = []

    for g1, g2 in combinations(groups, 2):
        p = data.loc[data[cluster_col] == g1, variable].value_counts(normalize=True)
        q = data.loc[data[cluster_col] == g2, variable].value_counts(normalize=True)

        tvd = total_variation_distance(p, q)
        mode_mismatch = float(p.idxmax() != q.idxmax())

        pair_results.append({
            "pair": f"({g1},{g2})",
            "tvd": tvd,
            "abs_tvd": tvd,  # already non-negative, kept for schema symmetry
            "mode_mismatch": mode_mismatch,
        })

    result = PairwiseVariableResult(variable=variable, pair_results=pair_results, var_kind="nominal")
    df = result.to_frame()
    result.mean_pairwise_effect = float(df["tvd"].mean())
    result.mean_secondary = float(df["mode_mismatch"].mean())
    return result

### Result container for one variable

In [ ]:
@dataclass
class PairwiseVariableResult:
    variable: str
    pair_results: list = field(default_factory=list)
    var_kind: Literal["ordinal", "nominal"] = "ordinal"
    mean_pairwise_effect: float = 0.0   # mean |Cliff's delta| (ordinal) or mean TVD (nominal) -- unit-free, aggregable
    mean_secondary: float = 0.0          # mean |median diff| (ordinal) or mean mode-mismatch rate (nominal)
    mean_tertiary: float = 0.0           # mean |mean diff| (ordinal only); unused for nominal

    def to_frame(self) -> pd.DataFrame:
        return pd.DataFrame(self.pair_results)

### |Cliff's δ| for every pair of clusters (ordered variable)

In [ ]:
def pairwise_separability_ordinal(
    data: pd.DataFrame,
    cluster_col: str,
    variable: str,
) -> PairwiseVariableResult:
    """
    For an ordinal/continuous variable: Cliff's delta, |median diff|, and
    |mean diff| for every cluster pair, averaged across all C(k,2) pairs.
    """
    groups = sorted(data[cluster_col].unique())
    pair_results = []

    for g1, g2 in combinations(groups, 2):
        x = data.loc[data[cluster_col] == g1, variable].dropna().to_numpy()
        y = data.loc[data[cluster_col] == g2, variable].dropna().to_numpy()

        delta = cliffs_delta(x, y)
        median_diff = abs(np.median(x) - np.median(y))
        mean_diff = abs(np.mean(x) - np.mean(y))

        pair_results.append({
            "pair": f"({g1},{g2})",
            "cliffs_delta": delta,
            "abs_cliffs_delta": abs(delta),
            "median_diff": median_diff,
            "mean_diff": mean_diff,
        })

    result = PairwiseVariableResult(variable=variable, pair_results=pair_results, var_kind="ordinal")
    df = result.to_frame()
    result.mean_pairwise_effect = float(df["abs_cliffs_delta"].mean())
    result.mean_secondary = float(df["median_diff"].mean())
    result.mean_tertiary = float(df["mean_diff"].mean())
    return result

### Result of one configuration

Per-variable values, their average (the aggregate PSI) and a text summary.

In [ ]:
@dataclass
class ConfigPairwiseSeparability:
    """
    Aggregated pairwise-separability result for a full clustering
    configuration, mirroring the existing ConfigSeparability report
    (epsilon-squared / Cramer's V) so both can sit side by side.
    """
    config_name: str
    per_variable: list = field(default_factory=list)  # list[PairwiseVariableResult]

    @property
    def pairwise_separability_index(self) -> float:
        """
        Unweighted mean of each variable's mean_pairwise_effect
        (mean |Cliff's delta| for ordinal vars, mean TVD for nominal vars).
        Both sub-metrics are unit-free and bounded in [0, 1], so they are
        safely averaged together across variable types.
        """
        if not self.per_variable:
            return float("nan")
        return float(np.mean([v.mean_pairwise_effect for v in self.per_variable]))

    def to_frame(self) -> pd.DataFrame:
        rows = []
        for v in self.per_variable:
            rows.append({
                "config": self.config_name,
                "variable": v.variable,
                "type": v.var_kind,
                "metric": "cliffs_delta" if v.var_kind == "ordinal" else "total_variation_distance",
                "mean_pairwise_effect": v.mean_pairwise_effect,
                "secondary_metric": "median_diff" if v.var_kind == "ordinal" else "mode_mismatch_rate",
                "mean_secondary": v.mean_secondary,
            })
        return pd.DataFrame(rows)

    @staticmethod
    def _effect_size_label(value: float) -> str:
        """Same qualitative thresholds as the existing separability report,
        applied here to |Cliff's delta| / TVD (both in [0,1])."""
        if value >= 0.474:   # Cliff's delta "large" threshold (Romano et al. 2006)
            return "large"
        elif value >= 0.33:
            return "medium"
        elif value >= 0.147:
            return "small"
        return "negligible"

    def summary(self) -> str:
        """Plain-text report, formatted like ConfigSeparability.summary()."""
        if not self.per_variable:
            return f"ConfigPairwiseSeparability('{self.config_name}'): no variables computed."

        rows = sorted(self.per_variable, key=lambda v: v.mean_pairwise_effect, reverse=True)
        name_w = max(len(v.variable) for v in rows) + 2

        header = (
            f"{'Variable':<{name_w}} {'Type':<9} {'Metric':<12} "
            f"{'MeanEff':>8} {'Effect':<11} {'Secondary':<14} {'Value':>8}"
        )
        sep = "-" * len(header)

        lines = [
            f"Pairwise Separability report — {self.config_name}",
            sep,
            header,
            sep,
        ]
        for v in rows:
            metric_name = "cliffs_delta" if v.var_kind == "ordinal" else "TVD"
            secondary_name = "median_diff" if v.var_kind == "ordinal" else "mode_mismatch"
            lines.append(
                f"{v.variable:<{name_w}} {v.var_kind:<9} {metric_name:<12} "
                f"{v.mean_pairwise_effect:>8.3f} {self._effect_size_label(v.mean_pairwise_effect):<11} "
                f"{secondary_name:<14} {v.mean_secondary:>8.3f}"
            )
        lines.append(sep)
        lines.append(
            f"{'PAIRWISE SEPARABILITY INDEX (mean)':<{name_w + 9 + 12}} "
            f"{self.pairwise_separability_index:>8.3f} "
            f"{self._effect_size_label(self.pairwise_separability_index):<11}"
        )
        return "\n".join(lines)

    def __repr__(self) -> str:
        return self.summary()

### Main function

Given the data, the cluster column and the variables, returns the PSI of the configuration.

In [ ]:
def compute_config_pairwise_separability(
    data: pd.DataFrame,
    cluster_col: str,
    ordinal_vars: Sequence[str] = (),
    nominal_vars: Sequence[str] = (),
    config_name: str = "",
) -> ConfigPairwiseSeparability:
    """
    Main entry point, analogous to compute_config_separability but for the
    pairwise (Cliff's delta / TVD) family of metrics.

    Example
    -------
    >>> result_I = compute_config_pairwise_separability(
    ...     data=df,
    ...     cluster_col="clust_label_I",
    ...     ordinal_vars=["political_stance_score", "argument_quality_score", "sentiment_score"],
    ...     nominal_vars=["discourse_tone_score", "dominant_frame_score"],
    ...     config_name="Config I",
    ... )
    >>> print(result_I.summary())
    """
    per_variable = []

    for var in ordinal_vars:
        per_variable.append(pairwise_separability_ordinal(data, cluster_col, var))

    for var in nominal_vars:
        per_variable.append(pairwise_separability_nominal(data, cluster_col, var))

    return ConfigPairwiseSeparability(config_name=config_name, per_variable=per_variable)